<a href="https://colab.research.google.com/github/AidoruFusion/northstar-databases-analytics/blob/main/Python_Cleaning_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: Python Data Cleaning and JSON Preparation

This notebook documents the first stage of the NorthStar data workflow.

The workflow follows this order:

- load the raw CSV files from Colab;
- inspect each file before cleaning;
- clean each dataset directly in its own section;
- save cleaned CSV files for SQL in R analysis;
- convert cleaned CSV files into JSON for MongoDB Atlas.

The purpose is to show how the raw NorthStar files were prepared before any database or analytics work was carried out.


# 1. Importing Python libraries

This stage prepares the notebook environment.

- `pandas` is used to load and clean CSV files as DataFrames.
- `numpy` is used to handle missing values such as `NaN`.
- `Path` is used to organise file locations.

These tools were used because the raw NorthStar data is stored in separate CSV files and must be cleaned before analysis.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 2. Setting the folder paths

This stage defines where the workflow starts and where the outputs are saved.

- Raw CSV files are read from the Colab working folder.
- Cleaned CSV files are saved in `/content/cleaned/csv`.
- JSON files are saved in `/content/cleaned/json`.

Keeping the outputs in separate folders makes the workflow easier to follow: raw data is kept separate from cleaned data and MongoDB-ready JSON files.


In [2]:
from pathlib import Path

raw_folder = Path("/content")
cleaned_csv_folder = Path("/content/cleaned/csv")
json_folder = Path("/content/cleaned/json")

cleaned_csv_folder.mkdir(parents=True, exist_ok=True)
json_folder.mkdir(parents=True, exist_ok=True)

print("Raw folder:", raw_folder)
print("Cleaned CSV folder:", cleaned_csv_folder)
print("JSON folder:", json_folder)


Raw folder: /content
Cleaned CSV folder: /content/cleaned/csv
JSON folder: /content/cleaned/json


# 3. Checking that the raw CSV files are available

This stage confirms that the required NorthStar files have been uploaded before cleaning begins.

This is important because the workflow depends on multiple files, including customers, orders, deliveries, drivers, vehicles, hubs, complaints, incidents, and app events. If any file is missing, later cleaning or relationship checks may fail.


In [3]:
csv_files = [
    "customers.csv",
    "orders.csv",
    "deliveries.csv",
    "drivers.csv",
    "vehicles.csv",
    "hubs.csv",
    "complaints.csv",
    "incidents.csv",
    "app_events.csv",
    "data_dictionary.csv"
]

for file_name in csv_files:
    file_path = raw_folder / file_name
    print(file_name, "exists:", file_path.exists())

customers.csv exists: True
orders.csv exists: True
deliveries.csv exists: True
drivers.csv exists: True
vehicles.csv exists: True
hubs.csv exists: True
complaints.csv exists: True
incidents.csv exists: True
app_events.csv exists: True
data_dictionary.csv exists: True


# 4. Inspecting the raw files before cleaning

This stage checks the condition of the raw datasets before changes are made.

The inspection looks at:

- row and column counts;
- missing values;
- duplicate records;
- data types;
- zone columns that may contain inconsistent values.

This provides evidence of what needed cleaning and avoids changing the data without first understanding its quality issues.


In [4]:
for file_name in csv_files:
    df = pd.read_csv(raw_folder / file_name)

    print("\n" + "=" * 70)
    print(file_name)
    print("Rows and columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nMissing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

    print("\nData types:")
    print(df.dtypes)

    zone_columns = [col for col in df.columns if "zone" in col.lower()]
    for col in zone_columns:
        print(f"\nUnique values in {col}:")
        print(sorted(df[col].dropna().astype(str).unique()))


customers.csv
Rows and columns: (650, 9)
Duplicate rows: 0

Missing values:
loyalty_score        20
preferred_channel    13
dtype: int64

Data types:
customer_id              object
age                       int64
home_zone                object
customer_type            object
signup_date              object
loyalty_score           float64
app_engagement_score    float64
preferred_channel        object
account_status           object
dtype: object

Unique values in home_zone:
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

orders.csv
Rows and columns: (1250, 11)
Duplicate rows: 0

Missing values:
booking_channel    25
dtype: int64

Data types:
order_id                  object
customer_id               object
service_type              object
order_created_at          object
promised_window_hours      int64
pickup_zone               object
dropoff_zone              object
priority

# 5. Cleaning issues identified

The inspection showed several issues that needed cleaning before analysis:

- some zone names were inconsistent, such as `Central`, `CENTRAL`, and `Ctr`;
- date columns were stored as text and needed conversion;
- numeric fields needed conversion before calculations could be performed;
- invalid values such as negative distances or impossible ratings needed to be treated carefully;
- delivery duration needed to be calculated from dispatch and completion times.

Cleaning these issues is necessary because NorthStar needs reliable analysis across zones, service types, deliveries, complaints, incidents, and app activity.


# 6. Zone standardisation mapping

This stage defines how inconsistent zone values are corrected.

For example:

- `Central`, `CENTRAL`, and `Ctr` are standardised to `Central`;
- `North`, `NORTH`, and `north` are standardised to `North`;
- `RiverSide` is standardised to `Riverside`.

This is needed because zone-level analysis would be inaccurate if the same location appeared under different labels.


In [5]:
zone_mapping = {
    "CENTRAL": "Central",
    "Central": "Central",
    "Ctr": "Central",
    "ctr": "Central",

    "NORTH": "North",
    "North": "North",
    "north": "North",

    "SOUTH": "South",
    "South": "South",
    "south": "South",

    "EAST": "East",
    "East": "East",
    "east": "East",

    "WEST": "West",
    "West": "West",
    "west": "West",

    "AIRPORT": "Airport",
    "Airport": "Airport",
    "airport": "Airport",

    "RiverSide": "Riverside",
    "Riverside": "Riverside",
    "riverside": "Riverside"
}

# 7. Cleaning customers.csv

This section cleans the customer dataset.

The cleaning focuses on:

- standardising the `home_zone` field;
- converting `signup_date` into a date format;
- converting age, loyalty score, and app engagement score into numeric values;
- treating unrealistic ages as missing.

This prepares the customer data for later links with orders, complaints, and service cases.


In [6]:
customers_raw = pd.read_csv(raw_folder / "customers.csv")
customers = customers_raw.copy()

display(customers.head())
print(customers.info())
print("Missing values before cleaning:")
print(customers.isnull().sum())
print("Duplicate rows before cleaning:", customers.duplicated().sum())

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61,AIRPORT,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75,CENTRAL,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           650 non-null    object 
 1   age                   650 non-null    int64  
 2   home_zone             650 non-null    object 
 3   customer_type         650 non-null    object 
 4   signup_date           650 non-null    object 
 5   loyalty_score         630 non-null    float64
 6   app_engagement_score  650 non-null    float64
 7   preferred_channel     637 non-null    object 
 8   account_status        650 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 45.8+ KB
None
Missing values before cleaning:
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel       13
account_status     

In [7]:
customers.columns = (
    customers.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

customers = customers.drop_duplicates()

text_columns = customers.select_dtypes(include="object").columns
for column in text_columns:
    customers[column] = customers[column].astype(str).str.strip()
    customers[column] = customers[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

customers["home_zone"] = customers["home_zone"].replace(zone_mapping)

customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers["loyalty_score"] = pd.to_numeric(customers["loyalty_score"], errors="coerce")
customers["app_engagement_score"] = pd.to_numeric(customers["app_engagement_score"], errors="coerce")

customers.loc[(customers["age"] < 16) | (customers["age"] > 100), "age"] = np.nan

display(customers.head())
customers.to_csv(cleaned_csv_folder / "customers_cleaned.csv", index=False)
print("customers_cleaned.csv saved successfully.")

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26.0,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61.0,Airport,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66.0,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75.0,Central,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26.0,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


customers_cleaned.csv saved successfully.


# 8. Cleaning orders.csv

This section cleans the orders dataset.

The cleaning focuses on:

- standardising pickup and drop-off zones;
- converting order creation dates;
- converting order value and promised delivery window into numeric values;
- removing invalid negative order values.

This is important because orders are the central link between customers, deliveries, complaints, and app events.


In [8]:
orders_raw = pd.read_csv(raw_folder / "orders.csv")
orders = orders_raw.copy()

display(orders.head())
print(orders.info())
print("Missing values before cleaning:")
print(orders.isnull().sum())
print("Duplicate rows before cleaning:", orders.duplicated().sum())

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               1250 non-null   object 
 1   customer_id            1250 non-null   object 
 2   service_type           1250 non-null   object 
 3   order_created_at       1250 non-null   object 
 4   promised_window_hours  1250 non-null   int64  
 5   pickup_zone            1250 non-null   object 
 6   dropoff_zone           1250 non-null   object 
 7   priority_level         1250 non-null   object 
 8   order_value            1250 non-null   float64
 9   booking_channel        1225 non-null   object 
 10  special_handling_flag  1250 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 107.6+ KB
None
Missing values before cleaning:
order_id                  0
customer_id               0
service_type              0
order_created_at          0
prom

In [9]:
orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

orders = orders.drop_duplicates()

text_columns = orders.select_dtypes(include="object").columns
for column in text_columns:
    orders[column] = orders[column].astype(str).str.strip()
    orders[column] = orders[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

orders["pickup_zone"] = orders["pickup_zone"].replace(zone_mapping)
orders["dropoff_zone"] = orders["dropoff_zone"].replace(zone_mapping)

orders["order_created_at"] = pd.to_datetime(orders["order_created_at"], errors="coerce")
orders["promised_window_hours"] = pd.to_numeric(orders["promised_window_hours"], errors="coerce")
orders["order_value"] = pd.to_numeric(orders["order_value"], errors="coerce")
orders["special_handling_flag"] = pd.to_numeric(orders["special_handling_flag"], errors="coerce")

orders.loc[orders["order_value"] < 0, "order_value"] = np.nan

display(orders.head())
orders.to_csv(cleaned_csv_folder / "orders_cleaned.csv", index=False)
print("orders_cleaned.csv saved successfully.")

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,Airport,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,Airport,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,Riverside,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,South,Low,125.58,Phone,0


orders_cleaned.csv saved successfully.


# 9. Cleaning deliveries.csv

This section cleans the deliveries dataset.

The cleaning focuses on:

- converting dispatch and completion times;
- converting delivery distance, route overrides, ratings, and cost fields into numeric values;
- calculating delivery duration in hours;
- treating negative durations, negative distances, and invalid ratings as missing.

This supports later analysis of delays, route performance, and customer experience.


In [10]:
deliveries_raw = pd.read_csv(raw_folder / "deliveries.csv")
deliveries = deliveries_raw.copy()

display(deliveries.head())
print(deliveries.info())
print("Missing values before cleaning:")
print(deliveries.isnull().sum())
print("Duplicate rows before cleaning:", deliveries.duplicated().sum())

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   delivery_id                    950 non-null    object 
 1   order_id                       950 non-null    object 
 2   driver_id                      950 non-null    object 
 3   vehicle_id                     950 non-null    object 
 4   hub_id                         950 non-null    object 
 5   dispatch_time                  950 non-null    object 
 6   delivery_completed_at          931 non-null    object 
 7   delivery_status                950 non-null    object 
 8   route_distance_km              950 non-null    float64
 9   manual_route_override_count    950 non-null    int64  
 10  proof_of_completion_missing    950 non-null    int64  
 11  customer_rating_post_delivery  936 non-null    float64
 12  fuel_or_charge_cost            950 non-null    flo

In [11]:
deliveries.columns = (
    deliveries.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

deliveries = deliveries.drop_duplicates()

text_columns = deliveries.select_dtypes(include="object").columns
for column in text_columns:
    deliveries[column] = deliveries[column].astype(str).str.strip()
    deliveries[column] = deliveries[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

deliveries["dispatch_time"] = pd.to_datetime(deliveries["dispatch_time"], errors="coerce")
deliveries["delivery_completed_at"] = pd.to_datetime(deliveries["delivery_completed_at"], errors="coerce")

numeric_columns = [
    "route_distance_km",
    "manual_route_override_count",
    "proof_of_completion_missing",
    "customer_rating_post_delivery",
    "fuel_or_charge_cost"
]

for column in numeric_columns:
    deliveries[column] = pd.to_numeric(deliveries[column], errors="coerce")

deliveries["delivery_duration_hours"] = (
    deliveries["delivery_completed_at"] - deliveries["dispatch_time"]
).dt.total_seconds() / 3600

deliveries.loc[deliveries["route_distance_km"] < 0, "route_distance_km"] = np.nan
deliveries.loc[deliveries["delivery_duration_hours"] < 0, "delivery_duration_hours"] = np.nan
deliveries.loc[
    (deliveries["customer_rating_post_delivery"] < 1) |
    (deliveries["customer_rating_post_delivery"] > 5),
    "customer_rating_post_delivery"
] = np.nan

display(deliveries.head())
deliveries.to_csv(cleaned_csv_folder / "deliveries_cleaned.csv", index=False)
print("deliveries_cleaned.csv saved successfully.")

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,delivery_duration_hours
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05,22.149973
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41,NaN
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51,1.108991
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62,23.985584
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22,4.042814


deliveries_cleaned.csv saved successfully.


# 10. Cleaning drivers.csv

This section cleans the drivers dataset.

The cleaning focuses on:

- standardising driver base zones;
- converting experience, training score, driver rating, and active status into numeric values;
- checking that training scores and ratings remain within valid ranges.

This prepares driver data for later comparison with delivery and incident performance.


In [12]:
drivers_raw = pd.read_csv(raw_folder / "drivers.csv")
drivers = drivers_raw.copy()

display(drivers.head())
print(drivers.info())
print("Missing values before cleaning:")
print(drivers.isnull().sum())
print("Duplicate rows before cleaning:", drivers.duplicated().sum())

,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,AIRPORT,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,AIRPORT,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,north,FullTime,3,69.7,4.14,Morning,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   driver_id         170 non-null    object 
 1   base_zone         170 non-null    object 
 2   employment_type   170 non-null    object 
 3   years_experience  170 non-null    int64  
 4   training_score    163 non-null    float64
 5   driver_rating     170 non-null    float64
 6   shift_preference  170 non-null    object 
 7   active_flag       170 non-null    int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 10.8+ KB
None
Missing values before cleaning:
driver_id           0
base_zone           0
employment_type     0
years_experience    0
training_score      7
driver_rating       0
shift_preference    0
active_flag         0
dtype: int64
Duplicate rows before cleaning: 0


In [13]:
drivers.columns = (
    drivers.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

drivers = drivers.drop_duplicates()

text_columns = drivers.select_dtypes(include="object").columns
for column in text_columns:
    drivers[column] = drivers[column].astype(str).str.strip()
    drivers[column] = drivers[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

drivers["base_zone"] = drivers["base_zone"].replace(zone_mapping)

drivers["years_experience"] = pd.to_numeric(drivers["years_experience"], errors="coerce")
drivers["training_score"] = pd.to_numeric(drivers["training_score"], errors="coerce")
drivers["driver_rating"] = pd.to_numeric(drivers["driver_rating"], errors="coerce")
drivers["active_flag"] = pd.to_numeric(drivers["active_flag"], errors="coerce")

drivers.loc[(drivers["training_score"] < 0) | (drivers["training_score"] > 100), "training_score"] = np.nan
drivers.loc[(drivers["driver_rating"] < 1) | (drivers["driver_rating"] > 5), "driver_rating"] = np.nan

display(drivers.head())
drivers.to_csv(cleaned_csv_folder / "drivers_cleaned.csv", index=False)
print("drivers_cleaned.csv saved successfully.")

,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,Airport,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,Airport,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,North,FullTime,3,69.7,4.14,Morning,1


drivers_cleaned.csv saved successfully.


# 11. Cleaning vehicles.csv

This section cleans the vehicles dataset.

The cleaning focuses on:

- standardising assigned zones;
- converting commission dates;
- converting battery health and odometer values into numeric values;
- checking that battery health stays between 0 and 100 and odometer readings are not negative.

This supports later analysis of fleet condition and vehicle-related operational issues.


In [14]:
vehicles_raw = pd.read_csv(raw_folder / "vehicles.csv")
vehicles = vehicles_raw.copy()

display(vehicles.head())
print(vehicles.info())
print("Missing values before cleaning:")
print(vehicles.isnull().sum())
print("Duplicate rows before cleaning:", vehicles.duplicated().sum())

,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928,Active,v2.2
1,V002,EV,AIRPORT,2024-04-21 16:14:00,67.9,159368,InRepair,v2.2
2,V003,CargoVan,north,2025-11-24 23:59:00,91.7,219359,Active,v2.1
3,V004,Hybrid,RiverSide,2024-06-07 13:21:00,NaN,36310,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638,Active,v2.2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   vehicle_id          120 non-null    object 
 1   vehicle_type        120 non-null    object 
 2   assigned_zone       120 non-null    object 
 3   commission_date     120 non-null    object 
 4   battery_health_pct  116 non-null    float64
 5   odometer_km         120 non-null    int64  
 6   maintenance_status  120 non-null    object 
 7   telematics_version  120 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 7.6+ KB
None
Missing values before cleaning:
vehicle_id            0
vehicle_type          0
assigned_zone         0
commission_date       0
battery_health_pct    4
odometer_km           0
maintenance_status    0
telematics_version    0
dtype: int64
Duplicate rows before cleaning: 0


In [15]:
vehicles.columns = (
    vehicles.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

vehicles = vehicles.drop_duplicates()

text_columns = vehicles.select_dtypes(include="object").columns
for column in text_columns:
    vehicles[column] = vehicles[column].astype(str).str.strip()
    vehicles[column] = vehicles[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

vehicles["assigned_zone"] = vehicles["assigned_zone"].replace(zone_mapping)

vehicles["commission_date"] = pd.to_datetime(vehicles["commission_date"], errors="coerce")
vehicles["battery_health_pct"] = pd.to_numeric(vehicles["battery_health_pct"], errors="coerce")
vehicles["odometer_km"] = pd.to_numeric(vehicles["odometer_km"], errors="coerce")

vehicles.loc[(vehicles["battery_health_pct"] < 0) | (vehicles["battery_health_pct"] > 100), "battery_health_pct"] = np.nan
vehicles.loc[vehicles["odometer_km"] < 0, "odometer_km"] = np.nan

display(vehicles.head())
vehicles.to_csv(cleaned_csv_folder / "vehicles_cleaned.csv", index=False)
print("vehicles_cleaned.csv saved successfully.")

,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928.0,Active,v2.2
1,V002,EV,Airport,2024-04-21 16:14:00,67.9,159368.0,InRepair,v2.2
2,V003,CargoVan,North,2025-11-24 23:59:00,91.7,219359.0,Active,v2.1
3,V004,Hybrid,Riverside,2024-06-07 13:21:00,NaN,36310.0,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638.0,Active,v2.2


vehicles_cleaned.csv saved successfully.


# 12. Cleaning hubs.csv

This section cleans the hubs dataset.

The cleaning focuses on:

- standardising hub zone names;
- converting capacity score into a numeric value;
- checking that capacity score stays within the expected 0 to 100 range.

This is relevant because the case study highlights underperforming city zones and hubs.


In [16]:
hubs_raw = pd.read_csv(raw_folder / "hubs.csv")
hubs = hubs_raw.copy()

display(hubs.head())
print(hubs.info())
print("Missing values before cleaning:")
print(hubs.isnull().sum())
print("Duplicate rows before cleaning:", hubs.duplicated().sum())

,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82
1,H02,South Link,South,Dispatch,78
2,H03,East Dock,East,Warehouse,74
3,H04,West Gate,West,Dispatch,69
4,H05,Central Core,Central,Control,88


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   hub_id          8 non-null      object
 1   hub_name        8 non-null      object
 2   zone            8 non-null      object
 3   hub_type        8 non-null      object
 4   capacity_score  8 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 452.0+ bytes
None
Missing values before cleaning:
hub_id            0
hub_name          0
zone              0
hub_type          0
capacity_score    0
dtype: int64
Duplicate rows before cleaning: 0


In [17]:
hubs.columns = (
    hubs.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

hubs = hubs.drop_duplicates()

text_columns = hubs.select_dtypes(include="object").columns
for column in text_columns:
    hubs[column] = hubs[column].astype(str).str.strip()
    hubs[column] = hubs[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

hubs["zone"] = hubs["zone"].replace(zone_mapping)
hubs["capacity_score"] = pd.to_numeric(hubs["capacity_score"], errors="coerce")
hubs.loc[(hubs["capacity_score"] < 0) | (hubs["capacity_score"] > 100), "capacity_score"] = np.nan

display(hubs.head())
hubs.to_csv(cleaned_csv_folder / "hubs_cleaned.csv", index=False)
print("hubs_cleaned.csv saved successfully.")

,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82.0
1,H02,South Link,South,Dispatch,78.0
2,H03,East Dock,East,Warehouse,74.0
3,H04,West Gate,West,Dispatch,69.0
4,H05,Central Core,Central,Control,88.0


hubs_cleaned.csv saved successfully.


# 13. Cleaning complaints.csv

This section cleans the complaints dataset.

The cleaning focuses on:

- converting complaint creation dates;
- converting resolution days and compensation amounts into numeric values;
- treating negative resolution or compensation values as invalid.

This prepares complaint data for later analysis of service dissatisfaction and customer experience.


In [18]:
complaints_raw = pd.read_csv(raw_folder / "complaints.csv")
complaints = complaints_raw.copy()

display(complaints.head())
print(complaints.info())
print("Missing values before cleaning:")
print(complaints.isnull().sum())
print("Duplicate rows before cleaning:", complaints.duplicated().sum())

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   complaint_id         320 non-null    object 
 1   customer_id          320 non-null    object 
 2   order_id             320 non-null    object 
 3   complaint_type       320 non-null    object 
 4   channel              320 non-null    object 
 5   severity             320 non-null    object 
 6   created_at           320 non-null    object 
 7   status               320 non-null    object 
 8   resolution_days      320 non-null    int64  
 9   compensation_amount  304 non-null    float64
dtypes: float64(1), int64(1), object(8)
memory usage: 25.1+ KB
None
Missing values before cleaning:
complaint_id            0
customer_id             0
order_id                0
complaint_type          0
channel                 0
severity                0
created_at              0
status       

In [19]:
complaints.columns = (
    complaints.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

complaints = complaints.drop_duplicates()

text_columns = complaints.select_dtypes(include="object").columns
for column in text_columns:
    complaints[column] = complaints[column].astype(str).str.strip()
    complaints[column] = complaints[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

complaints["created_at"] = pd.to_datetime(complaints["created_at"], errors="coerce")
complaints["resolution_days"] = pd.to_numeric(complaints["resolution_days"], errors="coerce")
complaints["compensation_amount"] = pd.to_numeric(complaints["compensation_amount"], errors="coerce")

complaints.loc[complaints["resolution_days"] < 0, "resolution_days"] = np.nan
complaints.loc[complaints["compensation_amount"] < 0, "compensation_amount"] = np.nan

display(complaints.head())
complaints.to_csv(cleaned_csv_folder / "complaints_cleaned.csv", index=False)
print("complaints_cleaned.csv saved successfully.")

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11.0,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4.0,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16.0,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7.0,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1.0,16.18


complaints_cleaned.csv saved successfully.


# 14. Cleaning incidents.csv

This section cleans the incidents dataset.

The cleaning focuses on:

- converting incident report dates;
- converting resolved hours into a numeric value;
- treating negative resolution times as invalid.

This prepares incident data for later links with deliveries and service failures.


In [20]:
incidents_raw = pd.read_csv(raw_folder / "incidents.csv")
incidents = incidents_raw.copy()

display(incidents.head())
print(incidents.info())
print("Missing values before cleaning:")
print(incidents.isnull().sum())
print("Duplicate rows before cleaning:", incidents.duplicated().sum())

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   incident_id        280 non-null    object 
 1   delivery_id        280 non-null    object 
 2   incident_type      280 non-null    object 
 3   reported_at        280 non-null    object 
 4   severity           280 non-null    object 
 5   resolution_status  280 non-null    object 
 6   resolved_hours     263 non-null    float64
dtypes: float64(1), object(6)
memory usage: 15.4+ KB
None
Missing values before cleaning:
incident_id           0
delivery_id           0
incident_type         0
reported_at           0
severity              0
resolution_status     0
resolved_hours       17
dtype: int64
Duplicate rows before cleaning: 0


In [21]:
incidents.columns = (
    incidents.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

incidents = incidents.drop_duplicates()

text_columns = incidents.select_dtypes(include="object").columns
for column in text_columns:
    incidents[column] = incidents[column].astype(str).str.strip()
    incidents[column] = incidents[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

incidents["reported_at"] = pd.to_datetime(incidents["reported_at"], errors="coerce")
incidents["resolved_hours"] = pd.to_numeric(incidents["resolved_hours"], errors="coerce")
incidents.loc[incidents["resolved_hours"] < 0, "resolved_hours"] = np.nan

display(incidents.head())
incidents.to_csv(cleaned_csv_folder / "incidents_cleaned.csv", index=False)
print("incidents_cleaned.csv saved successfully.")

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0


incidents_cleaned.csv saved successfully.


# 15. Cleaning app_events.csv

This section cleans the app events dataset.

The cleaning focuses on:

- standardising zone context values;
- converting event timestamps;
- converting API latency and success flags into numeric values;
- treating negative latency values as invalid.

Missing order IDs are kept because not every app event is expected to link to an order.


In [22]:
app_events_raw = pd.read_csv(raw_folder / "app_events.csv")
app_events = app_events_raw.copy()

display(app_events.head())
print(app_events.info())
print("Missing values before cleaning:")
print(app_events.isnull().sum())
print("Duplicate rows before cleaning:", app_events.duplicated().sum())

,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,north,301,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,SOUTH,60,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,CENTRAL,442,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,north,60,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_id         640 non-null    object
 1   customer_id      640 non-null    object
 2   order_id         496 non-null    object
 3   event_timestamp  640 non-null    object
 4   event_type       640 non-null    object
 5   session_id       640 non-null    object
 6   device_type      640 non-null    object
 7   zone_context     640 non-null    object
 8   api_latency_ms   640 non-null    int64 
 9   success_flag     640 non-null    int64 
dtypes: int64(2), object(8)
memory usage: 50.1+ KB
None
Missing values before cleaning:
event_id             0
customer_id          0
order_id           144
event_timestamp      0
event_type           0
session_id           0
device_type          0
zone_context         0
api_latency_ms       0
success_flag         0
dtype: int64
Duplicate rows before cl

In [23]:
app_events.columns = (
    app_events.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

app_events = app_events.drop_duplicates()

text_columns = app_events.select_dtypes(include="object").columns
for column in text_columns:
    app_events[column] = app_events[column].astype(str).str.strip()
    app_events[column] = app_events[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

app_events["zone_context"] = app_events["zone_context"].replace(zone_mapping)
app_events["event_timestamp"] = pd.to_datetime(app_events["event_timestamp"], errors="coerce")
app_events["api_latency_ms"] = pd.to_numeric(app_events["api_latency_ms"], errors="coerce")
app_events["success_flag"] = pd.to_numeric(app_events["success_flag"], errors="coerce")
app_events.loc[app_events["api_latency_ms"] < 0, "api_latency_ms"] = np.nan

display(app_events.head())
app_events.to_csv(cleaned_csv_folder / "app_events_cleaned.csv", index=False)
print("app_events_cleaned.csv saved successfully.")

,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,North,301.0,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,South,60.0,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118.0,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,Central,442.0,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,North,60.0,1


app_events_cleaned.csv saved successfully.


# 16. Cleaning data_dictionary.csv

This section applies light cleaning to the data dictionary.

The data dictionary is mainly used for documentation, so the focus is on:

- standardising column names;
- removing duplicate rows;
- cleaning extra spaces in text fields.

This keeps the dataset descriptions clear for the report and notebook workflow.


In [24]:
data_dictionary_raw = pd.read_csv(raw_folder / "data_dictionary.csv")
data_dictionary = data_dictionary_raw.copy()

display(data_dictionary.head())
print(data_dictionary.info())
print("Missing values before cleaning:")
print(data_dictionary.isnull().sum())
print("Duplicate rows before cleaning:", data_dictionary.duplicated().sum())

,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   file_name     9 non-null      object
 1   record_count  9 non-null      int64 
 2   description   9 non-null      object
dtypes: int64(1), object(2)
memory usage: 348.0+ bytes
None
Missing values before cleaning:
file_name       0
record_count    0
description     0
dtype: int64
Duplicate rows before cleaning: 0


In [25]:
data_dictionary.columns = (
    data_dictionary.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

data_dictionary = data_dictionary.drop_duplicates()

text_columns = data_dictionary.select_dtypes(include="object").columns
for column in text_columns:
    data_dictionary[column] = data_dictionary[column].astype(str).str.strip()
    data_dictionary[column] = data_dictionary[column].replace({"nan": np.nan, "None": np.nan, "": np.nan})

display(data_dictionary.head())
data_dictionary.to_csv(cleaned_csv_folder / "data_dictionary_cleaned.csv", index=False)
print("data_dictionary_cleaned.csv saved successfully.")

,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...


data_dictionary_cleaned.csv saved successfully.


# 17. Checking the cleaned CSV files

This stage confirms that all cleaned CSV files were created successfully.

These cleaned CSV outputs are important because they are used in the next stages of the project, especially SQL in R analysis and CSV-to-JSON conversion.


In [26]:
print("Cleaned CSV files:")
for file in sorted(cleaned_csv_folder.glob("*.csv")):
    print(file.name)

Cleaned CSV files:
app_events_cleaned.csv
complaints_cleaned.csv
customers_cleaned.csv
data_dictionary_cleaned.csv
deliveries_cleaned.csv
drivers_cleaned.csv
hubs_cleaned.csv
incidents_cleaned.csv
orders_cleaned.csv
vehicles_cleaned.csv


# 18. Creating the cleaning summary

This stage creates a summary table comparing the raw and cleaned files.

The table records:

- raw row and column counts;
- cleaned row and column counts;
- duplicate rows removed;
- missing values after cleaning.

This provides evidence that each dataset was processed and helps support the cleaning section of the report.


In [27]:
cleaned_dataframes = {
    "customers": customers,
    "orders": orders,
    "deliveries": deliveries,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs,
    "complaints": complaints,
    "incidents": incidents,
    "app_events": app_events,
    "data_dictionary": data_dictionary
}

raw_dataframes = {
    "customers": customers_raw,
    "orders": orders_raw,
    "deliveries": deliveries_raw,
    "drivers": drivers_raw,
    "vehicles": vehicles_raw,
    "hubs": hubs_raw,
    "complaints": complaints_raw,
    "incidents": incidents_raw,
    "app_events": app_events_raw,
    "data_dictionary": data_dictionary_raw
}

summary_rows = []

for name in cleaned_dataframes:
    raw_df = raw_dataframes[name]
    clean_df = cleaned_dataframes[name]

    summary_rows.append({
        "file_name": name + ".csv",
        "raw_rows": raw_df.shape[0],
        "cleaned_rows": clean_df.shape[0],
        "raw_columns": raw_df.shape[1],
        "cleaned_columns": clean_df.shape[1],
        "duplicates_removed": raw_df.duplicated().sum(),
        "missing_values_after_cleaning": clean_df.isnull().sum().sum()
    })

cleaning_summary = pd.DataFrame(summary_rows)
display(cleaning_summary)

cleaning_summary.to_csv(cleaned_csv_folder / "cleaning_summary.csv", index=False)
print("cleaning_summary.csv saved successfully.")

,file_name,raw_rows,cleaned_rows,raw_columns,cleaned_columns,duplicates_removed,missing_values_after_cleaning
0,customers.csv,650,650,9,9,0,33
1,orders.csv,1250,1250,11,11,0,25
2,deliveries.csv,950,950,13,14,0,116
3,drivers.csv,170,170,8,8,0,7
4,vehicles.csv,120,120,8,8,0,4
5,hubs.csv,8,8,5,5,0,0
6,complaints.csv,320,320,10,10,0,16
7,incidents.csv,280,280,7,7,0,17
8,app_events.csv,640,640,10,10,0,144
9,data_dictionary.csv,9,9,3,3,0,0


cleaning_summary.csv saved successfully.


# 19. Converting cleaned CSV files to JSON

This stage converts each cleaned CSV file into JSON format.

The cleaned CSV files are useful for SQL in R analysis, while JSON is useful for MongoDB because MongoDB stores data in a document-based format.

A small conversion function is kept here because the same CSV-to-JSON process needs to be repeated for each cleaned file.


In [28]:
def convert_csv_to_json(csv_file, json_folder):
    df = pd.read_csv(csv_file)

    json_file_name = csv_file.stem.replace("_cleaned", "") + ".json"
    json_path = json_folder / json_file_name

    df.to_json(
        json_path,
        orient="records",
        indent=4,
        date_format="iso"
    )

    print(f"{json_file_name} saved successfully.")


cleaned_csv_files = sorted(cleaned_csv_folder.glob("*_cleaned.csv"))

for csv_file in cleaned_csv_files:
    convert_csv_to_json(csv_file, json_folder)

print("All cleaned CSV files converted to JSON.")


app_events.json saved successfully.
complaints.json saved successfully.
customers.json saved successfully.
data_dictionary.json saved successfully.
deliveries.json saved successfully.
drivers.json saved successfully.
hubs.json saved successfully.
incidents.json saved successfully.
orders.json saved successfully.
vehicles.json saved successfully.
All cleaned CSV files converted to JSON.


# 20. Checking the JSON files

This stage confirms that JSON files were created successfully.

These JSON files are the outputs used for MongoDB Atlas upload in the next notebook.


In [29]:
print("JSON files:")
for file in sorted(json_folder.glob("*.json")):
    print(file.name)

JSON files:
app_events.json
complaints.json
customers.json
data_dictionary.json
deliveries.json
drivers.json
hubs.json
incidents.json
orders.json
vehicles.json


# 21. Loading the cleaned CSV files

This stage reloads the cleaned CSV files to confirm that the saved outputs can be used again.

This check is useful because the cleaned datasets need to be available for later analysis in R, SQL, and MongoDB preparation.


In [30]:
customers = pd.read_csv(cleaned_csv_folder / "customers_cleaned.csv")
orders = pd.read_csv(cleaned_csv_folder / "orders_cleaned.csv")
deliveries = pd.read_csv(cleaned_csv_folder / "deliveries_cleaned.csv")
drivers = pd.read_csv(cleaned_csv_folder / "drivers_cleaned.csv")
vehicles = pd.read_csv(cleaned_csv_folder / "vehicles_cleaned.csv")
hubs = pd.read_csv(cleaned_csv_folder / "hubs_cleaned.csv")
complaints = pd.read_csv(cleaned_csv_folder / "complaints_cleaned.csv")
incidents = pd.read_csv(cleaned_csv_folder / "incidents_cleaned.csv")
app_events = pd.read_csv(cleaned_csv_folder / "app_events_cleaned.csv")

print("Cleaned datasets loaded successfully.")

Cleaned datasets loaded successfully.


# 22. Final output check

This final check confirms that both output folders contain the expected files.

At this point, the workflow has moved from raw CSV files to cleaned CSV outputs and MongoDB-ready JSON files.


In [31]:
print("Cleaned CSV files:")
for file in sorted(cleaned_csv_folder.glob("*.csv")):
    print("-", file.name)

print("\nJSON files:")
for file in sorted(json_folder.glob("*.json")):
    print("-", file.name)

Cleaned CSV files:
- app_events_cleaned.csv
- cleaning_summary.csv
- complaints_cleaned.csv
- customers_cleaned.csv
- data_dictionary_cleaned.csv
- deliveries_cleaned.csv
- drivers_cleaned.csv
- hubs_cleaned.csv
- incidents_cleaned.csv
- orders_cleaned.csv
- vehicles_cleaned.csv

JSON files:
- app_events.json
- complaints.json
- customers.json
- data_dictionary.json
- deliveries.json
- drivers.json
- hubs.json
- incidents.json
- orders.json
- vehicles.json


# Workflow conclusion

This notebook completes the first stage of the project. The raw NorthStar CSV files were inspected, cleaned directly within each dataset section, saved as cleaned CSV outputs, and converted into JSON files for MongoDB.

The cleaned CSV files support SQL in R analysis, while the JSON files support MongoDB Atlas upload and NoSQL database development.
